In [1]:
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt
from itertools import combinations
from statsmodels.tsa.stattools import coint
from scipy.stats import ttest_rel

pd.set_option("display.width", 140)

In [2]:
UNIVERSE = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "LINK", "AVAX", "DOT", 
            "LTC", "ATOM", "AAVE", "UNI", "ARB", "OP", "SUI", "NEAR", "ALGO", "FIL", 
            "MATIC", "TRX", "BCH", "ETC", "XLM", "ICP", "APT", "SEI", "INJ", "RENDER", 
            "FET", "TIA", "HBAR", "VET", "THETA", "EOS", "SAND", "MANA", "GRT", "MKR"]
SYMBOLS = [f"{coin}/USD" for coin in UNIVERSE]

In [ ]:
symbols = [f"{coin}/USD" for coin in UNIVERSE if f"{coin}/USD" in markets]
pairs = list(combinations(symbols, 2))
print(f"{len(symbols)} coins available, {len(pairs)} candidate pairs")

37 coins available, 666 candidate pairs


In [ ]:
# ----------------------------------------------
# Format timestamp column; rename df columns function
# ----------------------------------------------

def to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close", "volume"]].copy()
    df = df.rename(columns={"close": symbol, "volume": f"{symbol}_volume"})
    return df

In [5]:
exchange = ccxt.coinbase({"enableRateLimit": True, "timeout": 30000})
markets = exchange.load_markets()

# ----------------------------------------------
# Fetch market data function
# ----------------------------------------------
def fetch_ohlcv_history(exchange, symbol, timeframe, since_ms, limit=300, max_retries=5):
    all_rows = []
    since = since_ms
    now_ms = exchange.milliseconds()

    while since < now_ms:
        batch = None
        for attempt in range(max_retries):
            try:
                batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
                break
            except (ccxt.RequestTimeout, ccxt.NetworkError) as e:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

        if not batch:
            break

        all_rows.extend(batch)
        since = batch[-1][0] + 1

    return all_rows

In [6]:
# ----------------------------------------------
# Fetch price data for each coin in universe
# ----------------------------------------------

YEARS_OF_HISTORY = 5 
TIMEFRAME = "1h"

since_dt = pd.Timestamp.utcnow() - pd.DateOffset(years=YEARS_OF_HISTORY)
HISTORY_SINCE = exchange.parse8601(since_dt.strftime("%Y-%m-%dT%H:%M:%SZ"))

prices = None
for coin in UNIVERSE:
    symbol = f"{coin}/USD"
    if symbol not in markets:
        continue
    data = fetch_ohlcv_history(exchange, symbol, timeframe=TIMEFRAME, since_ms=HISTORY_SINCE)
    if not data:
        continue
    df = to_df(data, coin)
    prices = df if prices is None else prices.merge(df, on="timestamp", how="inner")

prices = prices.sort_values("timestamp").reset_index(drop=True)
print(f"{len(prices)} aligned hourly bars, {prices['timestamp'].min()} -> {prices['timestamp'].max()}")

/var/folders/74/lqx4_3w16sz74ftgv_mn1ytr0000gn/T/ipykernel_59904/1113893799.py:8: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  since_dt = pd.Timestamp.utcnow() - pd.DateOffset(years=YEARS_OF_HISTORY)


37721 aligned hourly bars, 2021-08-19 22:00:00 -> 2025-12-10 19:00:00


In [43]:
# Keep the original 1h data so we can compare 4h / 8h / 12h later.
prices_1h = prices.copy()

# Main run stays at 4h, just like the original notebook.
prices = prices_1h.set_index("timestamp").resample("4h").agg({
    **{c: "last" for c in UNIVERSE if c in prices_1h.columns},
    **{f"{c}_volume": "sum" for c in UNIVERSE if f"{c}_volume" in prices_1h.columns},
}).dropna().reset_index()

In [44]:
# What coins had available data out of the original universe
available_assets = [c for c in UNIVERSE if c in prices.columns]
symbols = [f"{c}/USD" for c in available_assets]
pairs = list(combinations(symbols, 2))

In [45]:
# ----------------------------------------------
# Split price data into train and test (70/30)
# ----------------------------------------------
TRAIN_FRAC = 0.7

split_idx = int(len(prices) * TRAIN_FRAC)
train_prices = prices.iloc[:split_idx].reset_index(drop=True)
test_prices = prices.iloc[split_idx:].reset_index(drop=True)

print(f"Train: {train_prices['timestamp'].min()} -> {train_prices['timestamp'].max()}  ({len(train_prices)} bars)")
print(f"Test:  {test_prices['timestamp'].min()} -> {test_prices['timestamp'].max()}  ({len(test_prices)} bars)")

Train: 2021-08-19 20:00:00 -> 2024-08-25 08:00:00  (6610 bars)
Test:  2024-08-25 12:00:00 -> 2025-12-10 16:00:00  (2833 bars)


In [46]:
# ----------------------------------------------
# Align timestamp for price data of 2 assets
# ----------------------------------------------
def align_pair_prices(prices, asset_a, asset_b):
    cols = ["timestamp", asset_a, asset_b]
    vol_cols = [f"{asset_a}_volume", f"{asset_b}_volume"]
    needed = cols + [c for c in vol_cols if c in prices.columns]
    out = prices[needed].dropna().copy()
    out = out.sort_values("timestamp").reset_index(drop=True)
    return out

In [47]:
# ----------------------------------------------
# Regress log prices of assets to get alpha, beta, corr, and EG function
# ----------------------------------------------
def estimate_ols_spread(df, asset_a, asset_b):
    """
    Fit the hedge ratio via OLS (used ONLY on the train set) and run the
    Engle-Granger cointegration test on the two log-price series.
    Two nonstationary series can produce a highly significant OLS slope
    just because both trend -- that p-value is NOT a cointegration test,
    so we report the Engle-Granger p-value instead.
    """
    out = df.copy()
    out[f"log_{asset_a}"] = np.log(out[asset_a])
    out[f"log_{asset_b}"] = np.log(out[asset_b])

    X = sm.add_constant(out[f"log_{asset_b}"])
    y = out[f"log_{asset_a}"]
    model = sm.OLS(y, X).fit()
    
    alpha = model.params["const"]
    beta = model.params[f"log_{asset_b}"]
    corr = out[[f"log_{asset_a}", f"log_{asset_b}"]].corr().iloc[0, 1]
    out["spread"] = model.resid

    eg_stat, eg_pvalue, eg_crit = coint(out[f"log_{asset_a}"], out[f"log_{asset_b}"])

    return out, alpha, beta, corr, eg_pvalue

In [48]:
# ----------------------------------------------
# Compute spread on test data function
# ----------------------------------------------
def apply_fixed_spread(df, asset_a, asset_b, alpha, beta):
    """
    Build the spread on a NEW sample (e.g. the test set) using an alpha/beta
    that was already estimated elsewhere (the train set). No refitting here,
    so no look-ahead when this is used out-of-sample.
    """
    out = df.copy()
    out[f"log_{asset_a}"] = np.log(out[asset_a])
    out[f"log_{asset_b}"] = np.log(out[asset_b])
    out["spread"] = out[f"log_{asset_a}"] - (alpha + beta * out[f"log_{asset_b}"])
    return out

In [49]:
# ----------------------------------------------
# Compute spread halflife function
# ----------------------------------------------
def spread_half_life(spread):
    s = pd.Series(spread).dropna()
    lag = s.shift(1)
    diff = s - lag
    tmp = pd.concat([diff, lag], axis=1).dropna()
    tmp.columns = ["diff", "lag"]
    if len(tmp) < 5:
        return np.nan
    model = sm.OLS(tmp["diff"], sm.add_constant(tmp["lag"])).fit()
    lam = model.params["lag"]
    if lam >= 0:
        return np.nan
    return -np.log(2) / lam

In [50]:
# ----------------------------------------------
# Count number of Z-score crossings function
# ----------------------------------------------
def spread_crossings(z, threshold=2.0):
    z = pd.Series(z).dropna()
    if len(z) < 2:
        return 0
    upper = ((z.shift(1) < threshold) & (z >= threshold)).sum()
    lower = ((z.shift(1) > -threshold) & (z <= -threshold)).sum()
    return int(upper + lower)

In [51]:
# ----------------------------------------------
# Compute beta stability function
# ----------------------------------------------
def simple_beta_stability(df, asset_a, asset_b):
    """Compare the hedge ratio in the first half vs second half of training data."""
    mid = len(df) // 2
    if mid < 20:
        return np.nan

    def get_beta(x):
        la = np.log(x[asset_a])
        lb = np.log(x[asset_b])
        return sm.OLS(la, sm.add_constant(lb)).fit().params.iloc[1]

    b1 = get_beta(df.iloc[:mid])
    b2 = get_beta(df.iloc[mid:])
    if b1 == 0 or np.isnan(b1) or np.isnan(b2):
        return np.nan
    return abs(b2 - b1) / abs(b1)

In [52]:
# ----------------------------------------------
# Calculate and display pair summary function
# ----------------------------------------------
def pair_summary(prices, asset_a, asset_b, crossing_z=2.0):
    df, alpha, beta, corr, eg_pvalue = estimate_ols_spread(prices, asset_a, asset_b)

    spread = df["spread"].dropna()
    spread_vol = spread.std()
    hl = spread_half_life(spread)

    z = (spread - spread.mean()) / spread.std()
    crossings = spread_crossings(z, threshold=crossing_z)
    beta_stability = simple_beta_stability(df, asset_a, asset_b)

    return pd.Series({
        "Pair": f"{asset_a}/{asset_b}",
        "Asset_A": asset_a,
        "Asset_B": asset_b,
        "Corr": corr,
        "EG_Coint_Pvalue": eg_pvalue,
        "Alpha": alpha,
        "Beta": beta,
        "Spread_Vol": spread_vol,
        "Half_life": hl,
        "Z_score_Crossings": crossings,
        "Beta_Stability": beta_stability})

In [53]:
# ----------------------------------------------
# Select base pairs function
# Get pair summary for each candidate pair using train data
# Query results to filter unwanted pairs
# Result: Base list of viable pairs
# ----------------------------------------------
def select_base_pairs(prices_train, pairs, max_candidates=20):
    """
    Static pair screening for the TRAINING period only.

    Important:
    This is recalculated separately for each timeframe so that
    Alpha/Beta and all screening statistics belong to that timeframe.
    """
    candidate_rows = []

    for a, b in pairs:
        candidate_rows.append(pair_summary(prices_train, a.split("/")[0], b.split("/")[0]))

    pair_table = pd.DataFrame(candidate_rows)

    eligible = pair_table.query("Corr > 0.8")
    eligible = eligible.query("EG_Coint_Pvalue < 0.1")
    eligible = eligible.query("Half_life >= 12 and Half_life <= 600")
    eligible = eligible.query("Beta_Stability <= 0.35")

    eligible = (eligible.sort_values("Z_score_Crossings", ascending=False).reset_index(drop=True))

    return eligible.head(max_candidates).reset_index(drop=True)


In [54]:
# ----------------------------------------------
# Get base pairs
# ----------------------------------------------
# base_pairs = select_base_pairs(train_prices, pairs, max_candidates=50)
# print(f"{len(base_pairs)} pairs passed the statistical screen")
# base_pairs.head(10)


In [55]:
# ----------------------------------------------
# Compute rolling Z-score function
# ----------------------------------------------
def rolling_zscore(series, window=25):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std

In [56]:
# ----------------------------------------------
# Create Z-score positions per timestamp
# ----------------------------------------------
def build_zscore_positions(z, entry_z=2.0, exit_z=0.25, min_persist=4, cooldown_bars=0, allow_entry=None):
    position = 0
    cooldown = 0
    below_count = 0
    above_count = 0
    positions = []

    for i, val in enumerate(z):
        if pd.isna(val):
            positions.append(0)
            continue

        if val < -entry_z:
            below_count += 1
        else:
            below_count = 0

        if val > entry_z:
            above_count += 1
        else:
            above_count = 0

        if cooldown > 0:
            cooldown -= 1

        entry_ok = True if allow_entry is None else bool(allow_entry.iloc[i])

        if position == 0:
            if cooldown == 0 and entry_ok and below_count >= min_persist:
                position = 1
            elif cooldown == 0 and entry_ok and above_count >= min_persist:
                position = -1

        elif position == 1:
            if val > -exit_z:
                position = 0
                cooldown = cooldown_bars

        elif position == -1:
            if val < exit_z:
                position = 0
                cooldown = cooldown_bars

        positions.append(position)

    return pd.Series(positions, index=z.index)

In [57]:
# ----------------------------------------------
# Main backteset function - Z-score focus
# Align pair prices
# Apply fixed spread
# Apply low vollume filter
# Rolling Z-score
# Build Z-score positions
# Calculate metrics
# ----------------------------------------------
def backtest_pair_strategy(
    prices, asset_a, asset_b, alpha, beta, ann_factor_hourly,
    window, entry_z, exit_z, fee_bps,
    min_persist, cooldown_bars, vol_window):

    df = align_pair_prices(prices, asset_a, asset_b)
    df = apply_fixed_spread(df, asset_a, asset_b, alpha, beta)

    df["spread_mean"] = df["spread"].rolling(window).mean()
    df["spread_std"] = df["spread"].rolling(window).std()

    # Simple volume idea:
    # low relative volume -> 1.25x exposure
    # normal volume      -> 1.00x exposure
    # high volume        -> 0.50x exposure
    # very high volume   -> 0x exposure
    vol_a = f"{asset_a}_volume"
    vol_b = f"{asset_b}_volume"
    if vol_a in df.columns and vol_b in df.columns:
        rel_a = df[vol_a] / df[vol_a].rolling(vol_window).median()
        rel_b = df[vol_b] / df[vol_b].rolling(vol_window).median()
        df["relative_volume"] = (rel_a + rel_b) / 2
        vol_mean = df["relative_volume"].rolling(vol_window).mean()
        vol_std = df["relative_volume"].rolling(vol_window).std()
        df["volume_z"] = (df["relative_volume"] - vol_mean) / vol_std
        df["volume_scale"] = np.where(
            df["volume_z"] >= 2.0, 0.0,
            np.where(df["volume_z"] >= 1.0, 0.5,
                     np.where(df["volume_z"] <= -1.0, 1.25, 1.0)))
        df["volume_ok"] = df["volume_scale"] > 0
    else:
        df["volume_scale"] = 1.0
        df["volume_ok"] = True

    df["zscore"] = rolling_zscore(df["spread"], window=window)
    allow_entry = df["volume_ok"] 

    z_smooth = df["zscore"].rolling(3).mean()

    df["position"] = build_zscore_positions(
        z_smooth, entry_z=entry_z, exit_z=exit_z,
        min_persist=min_persist, cooldown_bars=cooldown_bars,
        allow_entry=allow_entry
    )

    # Adjust exposure using the volume signal.
    df["position"] = df["position"] * df["volume_scale"]

    gross = 1 + abs(beta)
    df["w_a"] = df["position"] / gross
    df["w_b"] = -beta * df["position"] / gross

    df[f"ret_{asset_a}"] = df[asset_a].pct_change()
    df[f"ret_{asset_b}"] = df[asset_b].pct_change()

    raw_ret = (
        df["w_a"].shift(1) * df[f"ret_{asset_a}"] +
        df["w_b"].shift(1) * df[f"ret_{asset_b}"]
    )
    df["raw_ret"] = raw_ret

    turnover = (
        df["w_a"].diff().abs().fillna(0) +
        df["w_b"].diff().abs().fillna(0)
    )
    df["turnover"] = turnover

    cost_per_turn = fee_bps / 10000.0
    df["strategy_ret"] = raw_ret - turnover * cost_per_turn
    gross_sharpe = (
        raw_ret.mean() / raw_ret.std()
    ) * ann_factor_hourly if raw_ret.std() != 0 and not np.isnan(raw_ret.std()) else np.nan

    df = df.set_index("timestamp")
    df["equity"] = (1 + df["strategy_ret"].fillna(0)).cumprod()

    r = df["strategy_ret"].dropna()
    sharpe = (r.mean() / r.std()) * ann_factor_hourly if r.std() not in (0,) and not np.isnan(r.std()) else np.nan
    max_dd = (df["equity"] / df["equity"].cummax() - 1).min()
    cum_ret = df["equity"].iloc[-1] - 1 if len(df) else np.nan
    trades = (df["position"].diff().fillna(0) != 0).sum()

    metrics = pd.Series({
        "Pair": f"{asset_a}/{asset_b}", 
        "Alpha": alpha, "Beta": beta, "Gross Sharpe": gross_sharpe, "Sharpe": sharpe,
        "Cumulative Return": cum_ret, "Max Drawdown": max_dd,
        "Trades": int(trades), "Avg Turnover/Bar": turnover.mean(),
        "Mean Hourly Return": r.mean(), "Vol Hourly": r.std(),
    })
    return df, metrics

In [58]:
# ----------------------------------------------
# Define hyperparameters
# Run pairs backtest function
# ----------------------------------------------
ANN_FACTOR_HOURLY = 6
FEE_BPS_MAIN = 20.0  
WINDOW = 25  
Z_ENTRY = 2.5
Z_EXIT = 0.5
MIN_PERSIST = 2
COOLDOWN_BARS = 0
VOL_WINDOW = 50

def run_pairs(prices_slice, selected_pairs, param_value, fee_bps, 
              window=WINDOW, exit_z=Z_EXIT, min_persist=MIN_PERSIST, 
              cooldown_bars=COOLDOWN_BARS, vol_window=VOL_WINDOW,):
    bts = {}
    ret_series = {}
    metrics_rows = []

    for _, row in selected_pairs.iterrows():
        a, b = row["Asset_A"], row["Asset_B"]
        kwargs = dict(
            window=window,
            fee_bps=fee_bps,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars,
            vol_window=vol_window)

        kwargs.update(entry_z=param_value, exit_z=exit_z)

        bt, m = backtest_pair_strategy(
            prices_slice, a, b,
            alpha=row["Alpha"], beta=row["Beta"], ann_factor_hourly=ANN_FACTOR_HOURLY,
            **kwargs)

        pair_name = f"{a}/{b}"
        bts[pair_name] = bt
        ret_series[pair_name] = bt["strategy_ret"].rename(pair_name)
        metrics_rows.append(m)

    return bts, ret_series, pd.DataFrame(metrics_rows)

In [59]:
# ----------------------------------------------
# Equal Volatility Weighting function
# ----------------------------------------------
def equal_vol_weights_from_returns(ret_series):
    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    vols = rets.std()
    inv_vol = 1 / vols.replace(0, np.nan)
    weights = inv_vol / inv_vol.sum()
    return weights, rets

# ----------------------------------------------
# More sophisticated weighting function
# Reduce weight of expensive or highly correlated spreads
# ----------------------------------------------
def quality_weights_from_returns(ret_series, metrics):
    """Simple upgrade: inverse-vol * net Sharpe / turnover / correlation."""
    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    vols = rets.std()
    inv_vol = 1 / vols.replace(0, np.nan)

    m = metrics.set_index("Pair")
    sharpe = m["Sharpe"].reindex(rets.columns).clip(-0.5, 0.5).fillna(0)
    turnover = m["Avg Turnover/Bar"].reindex(rets.columns).fillna(0)
    avg_corr = rets.corr().abs().mean().reindex(rets.columns).fillna(0)

    score = (inv_vol
        * (1 + sharpe)
        / (1 + turnover)
        / (1 + avg_corr))

    if score.sum() == 0:
        return equal_vol_weights_from_returns(ret_series)
    return score / score.sum(), rets


In [60]:
# ----------------------------------------------
# Apply weights to portfolio function
# ----------------------------------------------
def apply_fixed_weights(ret_series, weights):
    rets = pd.concat(ret_series.values(), axis=1, join="inner").dropna(how="any")
    w = weights.reindex(rets.columns).fillna(0.0)
    if w.sum() > 0:
        w = w / w.sum()   
    port_ret = rets.mul(w, axis=1).sum(axis=1)
    return port_ret, w, rets

In [61]:
# ----------------------------------------------
# Quarterly backtest function
# ----------------------------------------------
def quarterly_walkforward(prices, start_date, lookback,
    param_value=Z_ENTRY, fee_bps=FEE_BPS_MAIN, window=WINDOW,
    exit_z=Z_EXIT, max_pairs=20, min_persist=MIN_PERSIST, cooldown_bars=COOLDOWN_BARS):
    """
    Quarterly point-in-time pair selection and hedge-ratio estimation.

    At the beginning of each quarter:
      1. Take only trailing data available before the quarter.
      2. Re-screen all candidate pairs.
      3. Select the best pairs using only trailing data.
      4. Estimate alpha/beta on that trailing data.
      5. Trade the selected pairs during the following quarter
         using fixed alpha/beta and fixed portfolio weights.
    """

    prices = prices.sort_values("timestamp").reset_index(drop=True)

    results = []
    quarter_info = []
    weights_records = []
    pair_bts_all = {}

    test_prices = prices[prices["timestamp"] >= start_date].copy()

    # Calendar quarter containing each bar
    test_prices["quarter"] = test_prices["timestamp"].dt.to_period("Q")

    for quarter, quarter_data in test_prices.groupby("quarter"):

        quarter_start = quarter_data["timestamp"].min()

        # ---------------------------------------------------------
        # Only information available BEFORE this quarter
        # ---------------------------------------------------------
        historical = prices[prices["timestamp"] < quarter_start].copy()

        if len(historical) < lookback:
            continue

        trailing = historical.iloc[-lookback:].copy()

        # ---------------------------------------------------------
        # 1. Re-estimate pair statistics using trailing data only
        # ---------------------------------------------------------
        candidate_rows = []

        for a, b in pairs:
            candidate_rows.append(pair_summary(trailing, a.split("/")[0], b.split("/")[0]))

        pair_table_q = pd.DataFrame(candidate_rows)

        # Same filters as original strategy
        eligible = pair_table_q.query("Corr > 0.8")
        eligible = eligible.query("EG_Coint_Pvalue < 0.1")
        eligible = eligible.query("Half_life >= 12 and Half_life <= 600")
        eligible = eligible.query("Beta_Stability <= 0.35")

        eligible = eligible.sort_values("Z_score_Crossings",ascending=False).reset_index(drop=True)

        if len(eligible) == 0:
            continue

        # ---------------------------------------------------------
        # 2. Evaluate eligible pairs using ONLY trailing data
        # ---------------------------------------------------------
        train_bts, train_rets, train_metrics = run_pairs(
            trailing,
            eligible,
            param_value,
            fee_bps,
            window=window,
            exit_z=exit_z,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars)

        # Select best pairs using ONLY trailing information
        quality = (train_metrics[["Pair", "Sharpe", "Avg Turnover/Bar"]].sort_values("Sharpe", ascending=False).head(max_pairs))

        selected = eligible[eligible["Pair"].isin(quality["Pair"])].reset_index(drop=True)

        if len(selected) == 0:
            continue

        # ---------------------------------------------------------
        # 3. Calculate portfolio weights using trailing data
        # ---------------------------------------------------------
        selected_rets = {pair: train_rets[pair]
            for pair in train_rets
            if pair in selected["Pair"].values}

        selected_metrics = train_metrics[train_metrics["Pair"].isin(selected["Pair"])].copy()

        weights, _ = quality_weights_from_returns(selected_rets,selected_metrics)

        # Store weights for this quarter
        quarter_weights = weights.to_dict()
        weights_records.append({"Quarter": str(quarter), **quarter_weights})

        # ---------------------------------------------------------
        # 4. Trade following quarter using FIXED alpha/beta
        # ---------------------------------------------------------
        quarter_data_clean = quarter_data.drop(columns=["quarter"])

        quarter_bts, quarter_rets, quarter_metrics = run_pairs(
            quarter_data_clean,
            selected,
            param_value,
            fee_bps,
            window=window,
            exit_z=exit_z,
            min_persist=min_persist,
            cooldown_bars=cooldown_bars)

        # Save pair-level backtests so downstream turnover analysis can use them.
        for pair_name, bt in quarter_bts.items():
            pair_bts_all.setdefault(pair_name, []).append(bt)

        # Apply weights determined BEFORE the quarter began
        quarter_port_ret, applied_weights, _ = apply_fixed_weights(quarter_rets, weights)

        # Store quarterly portfolio returns
        results.append(quarter_port_ret)

        # Store quarter-level information
        quarter_info.append({
            "Quarter": str(quarter),
            "N_Eligible": len(eligible),
            "N_Selected": len(selected),
            "Selected_Pairs": selected["Pair"].tolist()
        })

    # -------------------------------------------------------------
    # No results
    # -------------------------------------------------------------
    if not results:
        return (
            pd.Series(dtype=float),
            pd.Series(dtype=float),
            pd.DataFrame(),
            pd.DataFrame(),
            {}
        )

    # -------------------------------------------------------------
    # Combine quarterly portfolio returns
    # -------------------------------------------------------------
    portfolio_returns = pd.concat(results).sort_index()

    # -------------------------------------------------------------
    # Create cumulative portfolio equity
    # -------------------------------------------------------------
    portfolio_equity = (1 + portfolio_returns).cumprod()

    # -------------------------------------------------------------
    # Create quarter-by-quarter weights DataFrame
    # -------------------------------------------------------------
    weights_df = pd.DataFrame(weights_records)

    if not weights_df.empty:
        weights_df = (
            weights_df
            .set_index("Quarter")
            .fillna(0.0)
        )

    # -------------------------------------------------------------
    # Quarter information DataFrame
    # -------------------------------------------------------------
    quarter_info = pd.DataFrame(quarter_info)

    # Combine each pair's quarterly backtests into one DataFrame
    combined_pair_bts = {}

    for pair_name, bt_list in pair_bts_all.items():
        combined_pair_bts[pair_name] = pd.concat(
            bt_list,
            axis=0
        ).sort_index()

    return portfolio_returns, portfolio_equity, weights_df, quarter_info, combined_pair_bts

In [62]:
# ----------------------------------------------
# Select Z-score pairs 
# ----------------------------------------------

# z_train_bts, z_train_rets, z_train_metrics = run_pairs(
#     train_prices, base_pairs, Z_ENTRY, FEE_BPS_MAIN,
#     min_persist=4, cooldown_bars=12)

# z_quality = z_train_metrics[["Pair", "Sharpe", "Avg Turnover/Bar"]].copy()
# z_quality = z_quality.sort_values("Sharpe", ascending=False)

# z_selected_pairs = base_pairs[base_pairs["Pair"].isin(z_quality.head(12)["Pair"])].reset_index(drop=True)

# print("Z-score selected pairs:")
# print(z_selected_pairs["Pair"].tolist())

In [63]:
# ----------------------------------------------
# Create weights (Equal Vol & Quality)
# ----------------------------------------------
# z_weights_equal, _ = equal_vol_weights_from_returns(z_train_rets)
# z_weights, _ = quality_weights_from_returns(z_train_rets, z_train_metrics)

# print("Selected pairs:")
# print(base_pairs[["Pair", "Half_life", "Beta_Stability", "Z_score_Crossings"]])
# print("\nZ-score quality weights:")
# print(z_weights.sort_values(ascending=False))

In [ ]:
# ----------------------------------------------
# Grid Search function
# ----------------------------------------------
def simple_grid_search(prices_1h, pairs, fee_bps=20.0):
    rows = []
    timeframes = [4, 8, 12]
    window_hours = [24, 48, 72, 120, 168, 240, 336]
    cooldown_bars_grid = [0, 4, 8, 12, 24]

    for hours in timeframes:
        p = prices_1h.set_index("timestamp").resample(f"{hours}h").agg({**{c: "last" for c in UNIVERSE if c in prices_1h.columns}, **{f"{c}_volume": "sum" for c in UNIVERSE if f"{c}_volume" in prices_1h.columns}}).dropna().reset_index()
        split = int(len(p) * TRAIN_FRAC)
        p_train = p.iloc[:split].reset_index(drop=True)
        timeframe_pairs = select_base_pairs(p_train, pairs)
        if timeframe_pairs.empty: continue
        print(f"{hours}h: {len(timeframe_pairs)} eligible training pairs")

        for wh in window_hours:
            window_bars = max(5, round(wh / hours))
            for cooldown_bars in cooldown_bars_grid:
                _, rets, _ = run_pairs(p_train, timeframe_pairs, Z_ENTRY, fee_bps, window=window_bars, min_persist=2, cooldown_bars=cooldown_bars)
                if not rets: continue
                rets = pd.concat(rets.values(), axis=1, join="inner").dropna(how="any")
                if rets.empty: continue
                inv_vol = 1 / rets.std().replace(0, np.nan)
                weights = inv_vol / inv_vol.sum()
                port_ret = rets.mul(weights, axis=1).sum(axis=1)
                if port_ret.std() == 0 or np.isnan(port_ret.std()): continue
                sharpe = (port_ret.mean() / port_ret.std()) * ANN_FACTOR_HOURLY
                rows.append({"Timeframe_h": hours, "Window_h": wh, "Window_bars": window_bars, "Cooldown_bars": cooldown_bars, "N_Pairs": len(timeframe_pairs), "Train_Sharpe": sharpe})

    return pd.DataFrame(rows).sort_values("Train_Sharpe", ascending=False).reset_index(drop=True)

z_grid_results = simple_grid_search(
    prices_1h,
    pairs,
    FEE_BPS_MAIN)

print("Z-Score: Best grid-search combinations:")
display(z_grid_results.head(10))

In [ ]:
# ----------------------------------------------
# Display best grid search parameters
# ----------------------------------------------
best_z = z_grid_results.iloc[0]

BEST_TIMEFRAME_Z = int(best_z["Timeframe_h"])
BEST_WINDOW_HOURS_Z = int(best_z["Window_h"])
BEST_COOLDOWN_Z = int(best_z["Cooldown_bars"])

print("Best timeframe Z-Score:", BEST_TIMEFRAME_Z, "hours")
print("Best window Z-Score:", BEST_WINDOW_HOURS_Z, "hours")
print("Best cooldown Z-Score:", BEST_COOLDOWN_Z, "hours")

Best timeframe Z-Score: 12 hours
Best window Z-Score: 168 hours
Best cooldown Z-Score: 24 hours


In [ ]:
# ----------------------------------------------
# Calculate portfolio metrics
# ----------------------------------------------
def portfolio_metrics(port_ret):

    port_ret = port_ret.dropna()

    daily_ret = ((1 + port_ret).resample("D").prod()- 1).dropna()

    equity = (1 + daily_ret).cumprod()

    sharpe = (daily_ret.mean()/daily_ret.std()) * np.sqrt(252)

    max_dd = (equity/equity.cummax() - 1).min()

    return {
        "Sharpe": sharpe,
        "Cumulative Return": equity.iloc[-1]-1,
        "Max Drawdown": max_dd,
        "N Days": len(daily_ret)
    }

In [ ]:
# ----------------------------------------------
# Obtain new price data depending on best parameters for Z-score
# ----------------------------------------------

prices_best_z = prices_1h.set_index("timestamp").resample(f"{BEST_TIMEFRAME_Z}h").agg({
    **{c: "last" for c in UNIVERSE if c in prices_1h.columns},
    **{f"{c}_volume": "sum" for c in UNIVERSE if f"{c}_volume" in prices_1h.columns},
}).dropna().reset_index()

split_idx_z = int(len(prices_best_z) * TRAIN_FRAC)

train_best_z = prices_best_z.iloc[:split_idx_z].reset_index(drop=True)
test_best_z = prices_best_z.iloc[split_idx_z:].reset_index(drop=True)

BEST_WINDOW_Z = max(5, round(BEST_WINDOW_HOURS_Z / BEST_TIMEFRAME_Z))

In [ ]:
# ----------------------------------------------
# Perform quarterly walkforward on new Z-score data
# ----------------------------------------------
LOOKBACK_Z = len(train_best_z)

first_z_date = (train_best_z["timestamp"].max().to_period("Q") + 1).start_time

z_port_ret, z_port_equity, z_w_applied, z_quarters, z_pair_bts = quarterly_walkforward(
    prices_best_z,
    start_date=first_z_date,
    lookback=LOOKBACK_Z,
    param_value=Z_ENTRY,
    fee_bps=FEE_BPS_MAIN,
    window=BEST_WINDOW_Z,
    exit_z=Z_EXIT,
    max_pairs=12,
    min_persist=1,
    cooldown_bars=BEST_COOLDOWN_Z
)

In [ ]:
print("Z-score quarterly selections:")
display(z_quarters)

rolling_summary = pd.DataFrame({"Z-score": portfolio_metrics(z_port_ret)}).T

rolling_summary

Z-score quarterly selections:


,Quarter,N_Eligible,N_Selected,Selected_Pairs
0,2024Q4,17,12,"[DOGE/ICP, DOT/FIL, DOGE/UNI, ICP/GRT, ETC/ICP..."
1,2025Q1,8,8,"[DOT/FIL, ETH/FET, FIL/MANA, UNI/ICP, BCH/MKR,..."
2,2025Q2,5,5,"[DOT/FIL, ETH/BCH, FIL/MANA, ICP/GRT, ETH/FET]"
3,2025Q3,2,2,"[DOT/FIL, EOS/MANA]"
4,2025Q4,4,4,"[DOT/FIL, ETC/ICP, ICP/GRT, DOGE/AAVE]"


,Sharpe,Cumulative Return,Max Drawdown,N Days
Z-score,-0.587936,-0.072511,-0.156209,436.0


In [42]:
def rolling_turnover_analysis(pair_bts_by_quarter, weights_df):
    rows = []
    for quarter, pair_bts in pair_bts_by_quarter.items():
        if quarter not in weights_df.index:
            continue
        weights = weights_df.loc[quarter].fillna(0.0)
        pair_turnover = pd.Series({
            pair: bt["turnover"].mean()
            for pair, bt in pair_bts.items()})

        common_pairs = pair_turnover.index.intersection(weights.index)

        weighted_turnover = (pair_turnover.loc[common_pairs]
            * weights.loc[common_pairs]).sum()

        rows.append({"Quarter": quarter,
            "Weighted Pair Turnover": weighted_turnover})

    return pd.DataFrame(rows).set_index("Quarter")


z_turnover_q = rolling_turnover_analysis(z_pair_bts, z_w_applied)

print("Z-score quarterly turnover:")
display(z_turnover_q)

print("Z-score average weighted pair turnover:",
    z_turnover_q["Weighted Pair Turnover"].mean())

KeyError: "None of ['Quarter'] are in the columns"

In [ ]:
def rolling_returns_at_fee(pair_bts_by_quarter, weights_df, fee_bps):
    quarterly_returns = []
    cost_per_turn = fee_bps / 10000.0

    for quarter, pair_bts in pair_bts_by_quarter.items():
        if quarter not in weights_df.index:
            continue
        weights = weights_df.loc[quarter].fillna(0.0)
        pair_returns = {}
        for pair_name, bt in pair_bts.items():
            if pair_name not in weights.index:
                continue
            pair_returns[pair_name] = (bt["raw_ret"] - bt["turnover"] * cost_per_turn).rename(pair_name)

        if not pair_returns:
            continue

        rets = pd.concat(pair_returns.values(), axis=1, join="inner").dropna(how="any")

        w = weights.reindex(rets.columns).fillna(0.0)

        if w.sum() > 0:
            w = w / w.sum()

        port_ret = rets.mul(w, axis=1).sum(axis=1)

        quarterly_returns.append(port_ret)

    if not quarterly_returns:
        return pd.Series(dtype=float)

    return pd.concat(quarterly_returns).sort_index()

In [ ]:
fee_grid = [0, 5, 7, 10, 15, 20, 25]
cost_rows = []

for fee in fee_grid:
    z_ret = rolling_returns_at_fee(z_pair_bts, z_w_applied, fee)

    cost_rows.append({"Strategy": "Z-score", "Fee_bps": fee, **portfolio_metrics(z_ret)})

cost_df = pd.DataFrame(cost_rows)
display(cost_df)

In [ ]:
def rolling_coin_exposure(pair_bts_by_quarter, weights_df):
    exposure_series = []

    for quarter, pair_bts in pair_bts_by_quarter.items():
        if quarter not in weights_df.index:
            continue
        weights = weights_df.loc[quarter].fillna(0.0)
        quarter_exposure = {}

        for pair_name, bt in pair_bts.items():
            w = weights.get(pair_name, 0.0)
            if w == 0:
                continue
            a, b = pair_name.split("/")
            quarter_exposure[a] = (quarter_exposure.get(a, 0.0) + (w * bt["w_a"].abs()))

            quarter_exposure[b] = (quarter_exposure.get(b, 0.0) + (w * bt["w_b"].abs()))

        if quarter_exposure:
            quarter_df = pd.DataFrame(quarter_exposure)
            exposure_series.append(quarter_df)

    if not exposure_series:
        return pd.DataFrame()

    all_exposure = pd.concat(exposure_series).sort_index()

    return all_exposure.mean().sort_values(ascending=False)

In [ ]:
z_coin_exposure = rolling_coin_exposure(z_pair_bts, z_w_applied)

coin_exp_df = pd.concat([z_coin_exposure.rename("Z-score")],axis=1).fillna(0)

coin_exp_df = coin_exp_df.sort_values("Z-score", ascending=False)

display(coin_exp_df.head(10))

ax = coin_exp_df.plot(kind="bar",figsize=(12, 5), width=0.8)

ax.set_title("Average Gross Coin Exposure Across Rolling Portfolio")
ax.set_ylabel("Average gross exposure")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("Z-score quarterly weights:")
display(z_w_applied.round(4))

print("\nAverage Z-score pair weights:")
display(z_w_applied.mean().sort_values(ascending=False).to_frame("Average Weight"))

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(z_port_equity,label="Z-score")

plt.title(f"Rolling Walk-Forward Out-of-Sample Equity ({FEE_BPS_MAIN:.0f} bps)")

plt.xlabel("Date")
plt.ylabel("Equity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()